# 02 — Select structures (RMSD)

Cluster a small set of NUDT15 PDB hits by CA-RMSD after superposition. The example pipeline still uses **5lpg** even if it shares a cluster with other structures.


In [ ]:
from pathlib import Path

import pandas as pd

from dualde.mapping import load_structural_mapping_json
from dualde.paths import example_dir
from dualde.rmsd import (
    plot_rmsd_cluster_counts,
    plot_rmsd_heatmap,
    rmsd_heatmap_after_alignment,
    summarize_rmsd_clusters_at_cutoff,
)

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
PDB_DIR = EXAMPLE / "pdb"

hits = pd.DataFrame(load_structural_mapping_json(EXAMPLE / "mapping" / "pdb_hits.json"))
print(hits.head())

# Small fixed list shipped under pdb/ (offline). Fetch more IDs from hits if needed.
EXAMPLE_STRUCTURE = "5lpg"
local_ids = ["5lpg", "5bon", "6t5j", "7b66"]
pdb_files = []
labels = []
chain_specs = {}
for pdb_id in local_ids:
    path = PDB_DIR / f"{pdb_id}.pdb"
    if not path.is_file():
        print("missing", path)
        continue
    pdb_files.append(str(path))
    labels.append(pdb_id)
    chain_specs[str(path)] = "A"

rmsd_df, cleaned = rmsd_heatmap_after_alignment(
    pdb_files=pdb_files,
    chain_specs=chain_specs,
    output_dir=str(PDB_DIR),
    labels=labels,
    show=False,
)
rmsd_df.to_csv(OUT / "rmsd_matrix.csv")
print(rmsd_df.round(2))


In [ ]:
CUTOFF = 2.0
summary = summarize_rmsd_clusters_at_cutoff(rmsd_df, cutoff=CUTOFF)
summary.to_csv(OUT / "rmsd_clusters.csv", index=False)
print(summary)
print("example structure:", EXAMPLE_STRUCTURE)

plot_rmsd_heatmap(rmsd_df, highlight_pdb=EXAMPLE_STRUCTURE)
plot_rmsd_cluster_counts(rmsd_df, cutoff=CUTOFF)
